# Langkah A (v2) — Scrape-by-Name PDKI dengan Query yang Benar
**Perbaikan berdasarkan analisis HAR file dari web PDKI**

Versi pertama gagal (127 dari 134 merek `TIDAK_KETEMU`) karena memakai struktur query yang salah. Analisis file HAR menunjukkan pencarian PDKI yang sebenarnya memakai:

- **`match_phrase` pada `nama_merek` dengan wildcard `*`** → mencari nama yang *diawali* frasa pencarian. Inilah yang membuat "Sari Temulawak Agung" berhasil menemukan "SARI TEMULAWAK AGUNG + LUKISAN TEMULAWAK".
- Beberapa field dibungkus `should` (id, nomor_permohonan, nomor_pendaftaran, owner, nama_merek).
- URL: `?keyword=<nama>&type=trademark`.
- **Tanpa filter tanggal** — supaya merek gold yang terdaftar sebelum 2016 (banyak kasus putusan) tetap ketemu. (Catat di metodologi: entri hasil injeksi bisa di luar rentang 2016–2026.)

Pendekatan: **nama-ternormalisasi + ambil-kandidat-teratas** — buang embel-embel "+ LUKISAN/LOGO" dan pisahan "A/B", lalu simpan semua kandidat teratas (yang mirip justru berguna sebagai *distractor*).


## 0. Setup

In [1]:
!pip install requests pandas tqdm rapidfuzz -q
import json, time, re, urllib.parse
import requests
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz
print("Setup selesai.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.9 MB/s eta 0:00:00
Setup selesai.


## 1. Konfigurasi request
Isi `COOKIE` dengan cookie sesi PDKI kamu (dari DevTools → Network → request `/search` → header `cookie`). `NEXT_ACTION` sudah sesuai HAR.


In [2]:
URL_BASE = "https://pdki-indonesia.dgip.go.id/search"

COOKIE = "PASTE_COOKIE_KAMU"
NEXT_ACTION = "87384f7de714212dd000e76fe09426dca6735260"
NEXT_ROUTER_STATE_TREE = "%5B%22%22%2C%7B%22children%22%3A%5B%22(with-header)%22%2C%7B%22children%22%3A%5B%22search%22%2C%7B%22children%22%3A%5B%22__PAGE__%22%2C%7B%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%5D%7D%2Cnull%2Cnull%2Ctrue%5D"

def build_headers(keyword):
    return {
        "accept": "text/x-component",
        "content-type": "text/plain;charset=UTF-8",
        "origin": "https://pdki-indonesia.dgip.go.id",
        "referer": f"https://pdki-indonesia.dgip.go.id/search?keyword={urllib.parse.quote(keyword)}&type=trademark",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
        "next-action": NEXT_ACTION,
        "next-router-state-tree": NEXT_ROUTER_STATE_TREE,
        "cookie": COOKIE,
    }

SIZE_PER_NAMA = 30    # kandidat teratas per nama
JEDA = 1.5            # detik antar-request

## 2. Parser JSON (sama seperti scraping kelasmu)

In [3]:
def extract_json_objects(text, marker='{"_index":"trademark"'):
    objects = []
    start = 0
    while True:
        idx = text.find(marker, start)
        if idx == -1:
            break
        brace_count = 0
        in_string = False
        escape = False
        end = idx
        for i in range(idx, len(text)):
            ch = text[i]
            if escape:
                escape = False
                continue
            if ch == "\\":
                escape = True
                continue
            if ch == '"':
                in_string = not in_string
                continue
            if not in_string:
                if ch == "{":
                    brace_count += 1
                elif ch == "}":
                    brace_count -= 1
                    if brace_count == 0:
                        end = i + 1
                        break
        try:
            objects.append(json.loads(text[idx:end]))
        except Exception:
            pass
        start = end
    return objects

## 3. Siapkan nama query (buang embel-embel, ambil bagian utama)

In [4]:
def siapkan_query(nama):
    """Bersihkan nama gold agar cocok dengan pencarian prefix PDKI.
    - Ambil bagian pertama bila ada "A/B" atau "A; B" (varian ganda dalam satu sel).
    - Buang penanda figuratif "+ LUKISAN/LOGO/GAMBAR" beserta ekornya.
    """
    s = str(nama)
    s = re.split(r"[;/]", s)[0]
    s = re.sub(r"\+\s*(lukisan|logo|gambar|device).*$", "", s, flags=re.I)
    s = re.sub(r"\s+", " ", s).strip()
    return s

## 4. Fungsi pencarian per-nama (query sesuai HAR)
Query meniru web PDKI: `match_phrase` pada `nama_merek` dengan wildcard, plus field lain di `should`. Hasil lalu **disaring** agar hanya menyisakan yang `nama_merek`-nya benar-benar mirip (mengurangi noise dari pencocokan nama pemilik).


In [5]:
def cari_merek_by_name(nama, size=SIZE_PER_NAMA):
    q = siapkan_query(nama)
    if not q:
        return []
    payload = [{
        "index": "trademark",
        "body": {
            "from": 0,
            "size": size,
            "query": {
                "bool": {
                    "must": [
                        {"bool": {"should": [
                            {"match": {"id": q}},
                            {"match": {"nomor_permohonan": q}},
                            {"match": {"nomor_pendaftaran": q}},
                            {"match_phrase": {"owner.tm_owner_name": q}},
                            {"match_phrase": {"nama_merek": q + "*"}},
                        ]}}
                    ],
                    "should": [],
                    "filter": []   # tanpa filter tanggal -> merek lama pun ketemu
                }
            },
            "sort": "$undefined",
        }
    }]
    url = f"{URL_BASE}?keyword={urllib.parse.quote(q)}&type=trademark"
    try:
        res = requests.post(url, headers=build_headers(q),
                            data=json.dumps(payload), timeout=30)
    except Exception as e:
        print(f"  ERROR koneksi '{nama}': {e}")
        return []
    if res.status_code != 200:
        print(f"  Gagal '{nama}' (status {res.status_code})")
        return []

    records = []
    for item in extract_json_objects(res.text):
        src = item.get("_source", {})
        nm = src.get("nama_merek", "") or ""
        owner = src.get("owner", [])
        owner_name = owner[0].get("tm_owner_name") if owner else None
        kelas = None
        tclass = src.get("t_class", [])
        if isinstance(tclass, list) and tclass:
            kl = [str(c.get("class_no")) for c in tclass if c.get("class_no")]
            kelas = ";".join(kl) if kl else None
        records.append({
            "id": src.get("id"),
            "kelas": kelas,
            "nama_merek": nm,
            "pemilik": owner_name,
            "nomor_permohonan": src.get("nomor_permohonan"),
            "tanggal_permohonan": src.get("tanggal_permohonan"),
            "status_permohonan": src.get("status_permohonan"),
            "query_asal": nama,
            "skor_match": fuzz.token_set_ratio(q.lower(), nm.lower()),
            "sumber": "injeksi",
        })
    # saring: buang hasil yg nama_merek-nya jauh (noise dari pencarian owner/nomor)
    records = [r for r in records if r["skor_match"] >= 55]
    records.sort(key=lambda r: r["skor_match"], reverse=True)
    return records

## 5. Muat daftar merek gold & jalankan

In [6]:
import os
DAFTAR_PATH = "merek_gold_untuk_scrape.csv"
if os.path.exists(DAFTAR_PATH):
    daftar = pd.read_csv(DAFTAR_PATH)["merek_gold"].dropna().astype(str).str.strip().tolist()
else:
    # fallback: ambil dari status_target_injeksi.csv atau gold cases
    for alt in ["status_target_injeksi.csv", "data_putusan_merek.csv"]:
        if os.path.exists(alt):
            dfa = pd.read_csv(alt, dtype=str).fillna("")
            if "merek_gold" in dfa.columns:
                daftar = dfa["merek_gold"].dropna().str.strip().tolist()
            else:
                daftar = sorted(set(dfa["merek_a"].str.strip()) | set(dfa["merek_b"].str.strip()))
                daftar = [d for d in daftar if d]
            break
    else:
        from google.colab import files
        print("Upload merek_gold_untuk_scrape.csv:")
        files.upload()
        daftar = pd.read_csv(DAFTAR_PATH)["merek_gold"].dropna().str.strip().tolist()
print(f"Jumlah nama merek gold: {len(daftar)}")

Upload merek_gold_untuk_scrape.csv:


Saving merek_gold_untuk_scrape.csv to merek_gold_untuk_scrape.csv
Jumlah nama merek gold: 134


## 6. Verifikasi: apakah target merek gold benar-benar ketemu?

In [10]:
def norm_cmp(s):
    s = str(s).lower()
    s = re.sub(r"\+\s*(lukisan|logo|gambar|device).*$", "", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

status = []
for nama in daftar:
    kand = df_injeksi[df_injeksi["query_asal"] == nama]["nama_merek"].dropna().tolist() if len(df_injeksi) else []
    skor = max([fuzz.ratio(norm_cmp(nama), norm_cmp(k)) for k in kand], default=0)
    status.append({
        "merek_gold": nama, "jml_kandidat": len(kand), "skor_terbaik": round(skor,1),
        "status": "KETEMU" if skor>=85 else "MIRIP_SAJA" if skor>=60 else "TIDAK_KETEMU"
    })
status_df = pd.DataFrame(status).sort_values("skor_terbaik")
print(status_df["status"].value_counts().to_string())
print("\nMasih TIDAK_KETEMU (kandidat injeksi manual dari dokumen putusan):")
display(status_df[status_df["status"]=="TIDAK_KETEMU"][["merek_gold","jml_kandidat"]])
status_df.to_csv("status_target_injeksi_v2.csv", index=False)

NameError: name 'df_injeksi' is not defined

## 7. Injeksi manual untuk yang tetap tidak ketemu
Merek yang tetap `TIDAK_KETEMU` (mis. merek asing yang tak terdaftar di Indonesia, atau sudah dihapus) disuntikkan **langsung dari nama di dokumen putusan**, agar retrieval punya target. Ditandai `sumber="injeksi_manual"`.


In [ ]:
tidak_ketemu = status_df[status_df["status"]=="TIDAK_KETEMU"]["merek_gold"].tolist()
manual_rows = [{
    "id": f"MANUAL_{i}", "kelas": "", "nama_merek": nm, "pemilik": "",
    "nomor_permohonan": "", "tanggal_permohonan": "", "status_permohonan": "",
    "query_asal": nm, "skor_match": 100, "sumber": "injeksi_manual"
} for i, nm in enumerate(tidak_ketemu)]
df_manual = pd.DataFrame(manual_rows)
print(f"Disuntik manual dari putusan: {len(df_manual)} merek")

## 8. Gabungkan ke haystack → `haystack_final_v2.csv`

In [ ]:
HAYSTACK_LAMA = "dataset_merek_2016_2026_300.csv"
if not os.path.exists(HAYSTACK_LAMA):
    from google.colab import files
    print(f"Upload {HAYSTACK_LAMA}:")
    files.upload()

hay_lama = pd.read_csv(HAYSTACK_LAMA, dtype=str, encoding="utf-8-sig").fillna("")
hay_lama.columns = [c.strip().lstrip("\ufeff") for c in hay_lama.columns]
hay_lama["query_asal"] = ""
hay_lama["skor_match"] = ""
hay_lama["sumber"] = "scraping_kelas"

kolom = ["id","kelas","nama_merek","pemilik","nomor_permohonan","tanggal_permohonan",
         "status_permohonan","query_asal","skor_match","sumber"]
for k in kolom:
    for d in (hay_lama, df_injeksi, df_manual):
        if k not in d.columns: d[k] = ""

gabung = pd.concat([hay_lama[kolom],
                    (df_injeksi[kolom].astype(str) if len(df_injeksi) else pd.DataFrame(columns=kolom)),
                    (df_manual[kolom].astype(str) if len(df_manual) else pd.DataFrame(columns=kolom))],
                   ignore_index=True)
gabung = gabung[gabung["nama_merek"].str.strip() != ""]
gabung = gabung.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

print(f"Haystack final: {len(gabung)} baris")
print(gabung["sumber"].value_counts().to_string())
gabung.to_csv("haystack_final_v2.csv", index=False, encoding="utf-8-sig")
print("\nTersimpan: haystack_final_v2.csv")

## 9. Unduh
```python
from google.colab import files
files.download("haystack_final_v2.csv")
files.download("status_target_injeksi_v2.csv")
```
Setelah ini, pakai `haystack_final_v2.csv` sebagai input Langkah B (preprocessing).
